# 10 · From frozen models to competition predictions

**Question:** Can the selected recommender produce every required prediction from only the official observed test prefixes?

**The corrected full run is complete and verified:** 1,671,803 sessions, 5,015,409 task rows, and 20 unique recommendations per row. Default mode replays eight official-prefix sessions against the actual full output. Full mode regenerates the artifact with frozen model weights and durable checkpoints.

The first submission used a post-competition release containing future events. Its 0.93554 public / 0.93583 private scores are invalidated and preserved as an incident record. Exact input provenance now guards inference. Kaggle accepted the corrected file and returned **0.56842 private / 0.56862 public**, marked **Complete (after deadline)**.


In [1]:
import hashlib
import json
import os
import subprocess
import time
from datetime import UTC, datetime
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display

STARTED = time.perf_counter()
FULL = os.environ.get("OTTO_FULL_INFERENCE") == "1"
location = Path.cwd().resolve()
ROOT = next(p for p in (location, *location.parents)
            if (p / "pyproject.toml").is_file()
            or ((p / "reports").is_dir() and (p / "configs").is_dir()))
print(datetime.now(UTC).isoformat(), "mode=", "full competition inference" if FULL else "verified native-model replay")


2026-09-10T06:48:37.118332+00:00 mode= verified native-model replay


## Model and data contract

Model weights are fixed by chronological model selection and evaluated before competition inference. Retrieval and historical statistics can then be refreshed from the official training events that precede the competition inputs. This refresh is a deployment operation; its outputs do not enter the reported temporal evaluation.

Set `OTTO_FULL_INFERENCE=1` in the managed execution environment to generate the complete prediction file. The managed run executes this mode with verified inputs, the locked project interpreter, and durable part checkpoints. Default replay requires no AWS access or training data.


In [2]:
if FULL:
    command = [str(ROOT / ".venv/bin/python"), str(ROOT / "scripts/run_inference.py"),
               "--stage", "predict", "--models", str(ROOT / "artifacts/research"),
               "--test", str(ROOT / "artifacts/test"), "--output", str(ROOT / "artifacts/inference"),
               "--threads", os.environ.get("OTTO_PREDICTION_THREADS", "4"),
               "--workers", os.environ.get("OTTO_PREDICTION_WORKERS", "1")]
    checkpoint_uri = os.environ.get("OTTO_PREDICTION_CHECKPOINT_URI")
    if checkpoint_uri:
        command += ["--checkpoint-uri", checkpoint_uri, "--owner-account", os.environ["OTTO_OWNER_ACCOUNT"],
                    "--region", os.environ["OTTO_AWS_REGION"]]
    completed = subprocess.run(command, cwd=ROOT, check=True, capture_output=True, text=True)
    print(completed.stdout)
    full = json.loads((ROOT / "artifacts/inference/prediction/manifest.json").read_text())
    destination = ROOT / "artifacts/inference/prediction/submission.csv.gz"
    preview = pd.read_csv(destination, nrows=9)
    assert full["status"] == "passed"
else:
    bundle = ROOT / "reports/research/inference_replay"
    manifest = json.loads((bundle / "manifest.json").read_text())
    assert manifest["status"] == "passed"
    for name, expected in manifest["files"].items():
        assert hashlib.sha256((bundle / name).read_bytes()).hexdigest() == expected, name
    candidates = pd.read_parquet(bundle / "features.parquet")
    records = []
    objectives = ("clicks", "carts", "orders")
    models = {o: lgb.Booster(model_file=str(bundle / manifest["models"][o]["path"])) for o in objectives}
    for session, query in candidates.groupby("session", sort=True):
        aids = query["aid"].to_numpy()
        for objective in objectives:
            names = manifest["models"][objective]["features"]
            assert models[objective].feature_name() == names
            score = models[objective].predict(query[names].to_numpy(dtype=np.float32), num_threads=1)
            order = np.lexsort((aids, -score))[:20]
            records.append((f"{session}_{objective}", " ".join(map(str, aids[order]))))
    replay = pd.DataFrame(records, columns=["session_type", "labels"])
    destination = ROOT / "artifacts/inference_replay.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    replay.to_csv(destination, index=False)
    assert destination.read_bytes() == (bundle / "expected.csv").read_bytes()
    full = manifest["full_prediction"]
    preview = replay.head(9)
    print(f"Exact replay passed: {len(replay):,} rows, {manifest['sessions']} real competition sessions")


Exact replay passed: 24 rows, 8 real competition sessions


## Coverage and output validation

Every observed session must appear exactly once for each of clicks, carts, and orders. Each recommendation list contains 20 unique nonnegative item IDs. The full validator checks the exact session ledger, objective coverage, duplicate rows, list lengths, and the final file checksum. Partial or incompatible artifacts cannot certify completion.


In [3]:
display(pd.DataFrame([
    ("Complete competition sessions", f"{full['sessions']:,}"),
    ("Required task rows", f"{full['rows']:,}"),
    ("Prediction artifact SHA-256", full["sha256"]),
    ("Prediction input identity", full["input_id"]),
], columns=["Verified evidence", "Value"]))
assert full["rows"] == 3 * full["sessions"]
for labels in preview["labels"]:
    items = labels.split()
    assert len(items) == len(set(items)) == 20
    assert all(item.isdecimal() for item in items)
display(preview)

# Bind the published delivery state to the exact replayed artifact.
submission = json.loads((ROOT / "reports/submissions/kaggle_submission.json").read_text())
receipt_body = {k: v for k, v in submission.items() if k != "receipt_id"}
assert hashlib.sha256(json.dumps(receipt_body, sort_keys=True, separators=(",", ":"),
                                allow_nan=False).encode()).hexdigest() == submission["receipt_id"]
assert submission["prediction"]["sha256"] == full["sha256"]
for name, expected in submission["source_files"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == expected, name
contract = json.loads((ROOT / "reports/research/competition_prediction_contract.json").read_text())
attestation = json.loads((ROOT / "reports/submissions/competition_input.json").read_text())
assert contract["competition_input"]["raw_sha256"] == attestation["raw_sha256"]
display(pd.DataFrame([
    ("Submission state", submission["status"]),
    ("Public score", submission["displayed_scores"]["public"] or "Pending"),
    ("Private score", submission["displayed_scores"]["private"] or "Pending"),
    ("Official observed test events", f"{contract['competition_input']['events']:,}"),
], columns=["Kaggle delivery", "Verified evidence"]))
if submission["status"] != "complete":
    assert all(value is None for value in submission["displayed_scores"].values())
    print("File uploaded; the final Kaggle Submit action is awaiting confirmation. No score claimed.")
else:
    assert submission["valid_competition_evaluation"] is True
    assert all(value is not None for value in submission["displayed_scores"].values())
    print("Kaggle accepted and scored this exact official-prefix artifact after the deadline.")
historical = json.loads((ROOT / submission["historical_invalidated_receipt"]).read_text())
assert historical["valid_competition_evaluation"] is False
print("Historical private score 0.93583 is invalidated; it is not evidence of model quality.")


,Verified evidence,Value
0,Complete competition sessions,"1,671,803"
1,Required task rows,"5,015,409"
2,Prediction artifact SHA-256,49e332948b47396fb609eaaac09a4cf3089468c38b58a1...
3,Prediction input identity,0dafdd7d6ee7a3680a1b89d0166eccd09a84b35dc68b51...


,session_type,labels
0,12899779_clicks,59625 737445 731692 1700255 941596 1253524 140...
1,12899779_carts,59625 731692 737445 941596 1340695 1422133 170...
2,12899779_orders,59625 731692 737445 1340695 941596 1422133 475...
3,12899780_clicks,1142000 736515 582732 973453 1502122 889686 48...
4,12899780_carts,1142000 582732 736515 973453 1758603 1502122 4...
5,12899780_orders,1142000 736515 582732 973453 1758603 487136 12...
6,12899781_clicks,199008 918667 532314 644935 91240 820257 27362...
7,12899781_carts,918667 199008 205149 428697 1714342 1736107 53...
8,12899781_orders,199008 918667 428697 532314 1714342 205149 644...


,Kaggle delivery,Verified evidence
0,Submission state,complete
1,Public score,0.56862
2,Private score,0.56842
3,Official observed test events,"6,928,123"


Kaggle accepted and scored this exact official-prefix artifact after the deadline.
Historical private score 0.93583 is invalidated; it is not evidence of model quality.


## Download the verified full submission

The complete file is `submission.csv.gz`, **289,541,110 bytes**, with SHA-256 `49e332948b47396fb609eaaac09a4cf3089468c38b58a15203075d6c107260f2`. [Download instructions](../docs/INFERENCE.md#download-and-submit-the-completed-full-run) copy the exact version from durable S3 storage and check its digest.

The default replay CSV contains only 24 rows and must not be submitted. The full gzip was independently downloaded, rehashed, and validated against the official session ledger. A full-mode notebook execution receipt records 2,413.288 seconds. No retraining is required to use this artifact.


## Decision and limits

The official input contains 6,928,123 prefix events across 1,671,803 sessions. A chronological train/test separation alone did not catch the original error because future events were embedded inside each full test session. Source provenance and byte-level input attestation now guard the competition path.

The feature-gain evidence remains the controlled temporal study in Notebook 09. Regenerating predictions does not refit or select model weights. The independent Kaggle submission of this exact file returned **0.56842 private / 0.56862 public**. This late result uses a different evaluation set from the temporal study and does not establish an official leaderboard rank or state-of-the-art performance.


In [4]:
print(datetime.now(UTC).isoformat(), "inference_notebook_complete",
      f"elapsed_seconds={time.perf_counter()-STARTED:.3f}")
print("Full competition submission:" if FULL else "Review example only (24 rows; do not upload):", destination)
print("Kaggle upload is a separate action; this notebook does not submit automatically.")
print("Runtime:", {"lightgbm": lgb.__version__, "numpy": np.__version__, "pandas": pd.__version__})


2026-09-10T06:48:37.257918+00:00 inference_notebook_complete elapsed_seconds=0.140
Review example only (24 rows; do not upload): /tmp/otto-notebooks-7iz2rk_k/artifacts/inference_replay.csv
Kaggle upload is a separate action; this notebook does not submit automatically.
Runtime: {'lightgbm': '4.7.0', 'numpy': '2.5.3', 'pandas': '3.0.5'}
